# Lexical Processing Deep Dive (SNLP Note 02)

Companion notebook for `02-Lexical-Processing-Part-1.md`. Every number quoted in the note's deep-dive sections, worked examples and practice solutions comes from a cell here. Uses only NLTK, numpy and matplotlib (no GPU, no downloads beyond NLTK data).

| Part | Topic |
|---|---|
| A | Zipf's and Heaps' laws: fits on Gutenberg texts |
| B | Types, tokens and TTR under different normalisations |
| C | Porter stemmer by hand: measure m, step-by-step trace, check against NLTK |
| D | Porter vs Snowball vs Lancaster; over- and under-stemming |
| E | POS-dependent lemmatization |
| F | BPE trainer with merge log; WordPiece score; Unigram (Viterbi) segmentation; language disparity |
| G | Levenshtein distance with backtrace; a tiny spelling corrector |
| H | Unicode normalisation (NFC/NFD/NFKC/NFKD) incl. Devanagari |
| I | A regex tokenizer with unit tests |
| J | Tricky sentence segmentation |

In [1]:
import re, unicodedata
from collections import Counter
import numpy as np
import nltk
for p in ["punkt", "punkt_tab", "wordnet", "omw-1.4", "averaged_perceptron_tagger_eng", "gutenberg", "stopwords"]:
    nltk.download(p, quiet=True)
from nltk.corpus import gutenberg, stopwords, wordnet as wn
from nltk.stem import PorterStemmer, SnowballStemmer, LancasterStemmer, WordNetLemmatizer
import matplotlib.pyplot as plt

def tokens(fid):
    "lower-case alphabetic tokens: the Python equivalent of tr -sc 'A-Za-z' '\\n' | tr 'A-Z' 'a-z'"
    return re.findall(r"[a-z]+", gutenberg.raw(fid).lower())

## Part A — Zipf's law and Heaps' law

Zipf: $f(r) \approx C / r^{s}$ with $s \approx 1$, so $\log f = \log C - s \log r$ is a straight line. We estimate $s$ by least squares on the log–log points (ranks 1–1000; the long tail of frequency-1 words forms flat steps that would bias the fit).

Heaps: $V(N) \approx k N^{\beta}$, so $\log V = \log k + \beta \log N$. We record the vocabulary size after every 1000 tokens and fit a line.

In [2]:
TEXTS = ["carroll-alice.txt", "austen-emma.txt", "melville-moby_dick.txt"]
fits = {}
for fid in TEXTS:
    t = tokens(fid); c = Counter(t)
    f = np.array(sorted(c.values(), reverse=True)); r = np.arange(1, len(f) + 1)
    s, b = np.polyfit(np.log10(r[:1000]), np.log10(f[:1000]), 1)
    seen, Ns, Vs = set(), [], []
    for i, w in enumerate(t, 1):
        seen.add(w)
        if i % 1000 == 0: Ns.append(i); Vs.append(len(seen))
    beta, lk = np.polyfit(np.log(Ns), np.log(Vs), 1)
    hapax = sum(v == 1 for v in c.values())
    fits[fid] = (len(t), len(c), -s, 10**b, beta, np.exp(lk))
    print(f"{fid:24s} N={len(t):6d} V={len(c):5d}  Zipf s={-s:.3f} C={10**b:8.1f} | "
          f"Heaps beta={beta:.3f} k={np.exp(lk):.2f} | hapax={hapax} ({hapax/len(c):.1%} of types)")

carroll-alice.txt        N= 27335 V= 2569  Zipf s=1.114 C=  6807.4 | Heaps beta=0.576 k=7.27 | hapax=1114 (43.4% of types)
austen-emma.txt          N=161975 V= 7096  Zipf s=1.136 C= 41948.1 | Heaps beta=0.522 k=14.24 | hapax=2751 (38.8% of types)
melville-moby_dick.txt   N=218382 V=16950  Zipf s=1.072 C= 35079.0 | Heaps beta=0.609 k=10.11 | hapax=7385 (43.6% of types)


In [3]:
# Zipf check on Alice: if f = C/r exactly, then r*f would be constant.
t = tokens("carroll-alice.txt"); c = Counter(t); N = len(t)
print(f"N = {N}")
for r_, (w, f_) in enumerate(c.most_common(10), 1):
    print(f"rank {r_:2d}  {w:5s} f={f_:5d}  r*f={r_*f_:5d}  share={f_/N:.4f}")

N = 27335
rank  1  the   f= 1642  r*f= 1642  share=0.0601
rank  2  and   f=  872  r*f= 1744  share=0.0319
rank  3  to    f=  729  r*f= 2187  share=0.0267
rank  4  a     f=  632  r*f= 2528  share=0.0231
rank  5  it    f=  595  r*f= 2975  share=0.0218
rank  6  she   f=  553  r*f= 3318  share=0.0202
rank  7  i     f=  545  r*f= 3815  share=0.0199
rank  8  of    f=  514  r*f= 4112  share=0.0188
rank  9  said  f=  462  r*f= 4158  share=0.0169
rank 10  you   f=  411  r*f= 4110  share=0.0150


In [4]:
# How much text do the most frequent words cover? (why stop-word removal shrinks indexes so much)
t = tokens("melville-moby_dick.txt"); c = Counter(t); N = len(t)
cum = np.cumsum(sorted(c.values(), reverse=True)) / N
for k in [10, 50, 100, 1000]:
    print(f"top {k:5d} types cover {cum[k-1]:.1%} of Moby Dick tokens")
sw = set(stopwords.words("english"))
print(f"NLTK English stop list: {len(sw)} words; they make up {sum(c[w] for w in sw)/N:.1%} of Moby Dick tokens")

top    10 types cover 23.5% of Moby Dick tokens
top    50 types cover 42.5% of Moby Dick tokens
top   100 types cover 51.3% of Moby Dick tokens
top  1000 types cover 76.7% of Moby Dick tokens
NLTK English stop list: 198 words; they make up 49.4% of Moby Dick tokens


In [5]:
# Heaps extrapolation: Moby Dick, fitted parameters
N_, V_, s_, C_, beta, k = fits["melville-moby_dick.txt"]
for n in [10_000, 100_000, N_, 2 * N_]:
    print(f"N={n:7d}: predicted V = {k * n**beta:8.0f}")
print("actual V at full length:", V_)

N=  10000: predicted V =     2761
N= 100000: predicted V =    11226
N= 218382: predicted V =    18064
N= 436764: predicted V =    27553
actual V at full length: 16950


## Part B — Types, tokens and TTR under three (four) normalisations

In [6]:
text = "The runner runs. The Runners ran; a runner's run is running, and RUNNING runs!"
P = PorterStemmer()
L = WordNetLemmatizer()
n1 = text.split()                                                     # N1: raw whitespace split
n2 = [w.replace("'s", "") for w in re.findall(r"[a-z]+(?:'[a-z]+)?", text.lower())]  # N2: lower + strip punct + drop 's
n3 = [P.stem(w) for w in n2]                                          # N3: N2 + Porter
pos = {"N": wn.NOUN, "V": wn.VERB, "J": wn.ADJ, "R": wn.ADV}
n4 = [L.lemmatize(w, pos.get(tag[0], wn.NOUN)) for w, tag in nltk.pos_tag(n2)]  # N4: N2 + POS lemmatizer
for name, toks in [("N1 raw split", n1), ("N2 lower+punct", n2), ("N3 +Porter", n3), ("N4 +lemma(POS)", n4)]:
    print(f"{name:15s} tokens={len(toks):2d} types={len(set(toks)):2d} TTR={len(set(toks))/len(toks):.3f}  {sorted(set(toks))}")
print(nltk.pos_tag(n2))

N1 raw split    tokens=14 types=13 TTR=0.929  ['RUNNING', 'Runners', 'The', 'a', 'and', 'is', 'ran;', 'run', 'runner', "runner's", 'running,', 'runs!', 'runs.']
N2 lower+punct  tokens=14 types=10 TTR=0.714  ['a', 'and', 'is', 'ran', 'run', 'runner', 'runners', 'running', 'runs', 'the']
N3 +Porter      tokens=14 types= 7 TTR=0.500  ['a', 'and', 'is', 'ran', 'run', 'runner', 'the']
N4 +lemma(POS)  tokens=14 types= 6 TTR=0.429  ['a', 'and', 'be', 'run', 'runner', 'the']
[('the', 'DT'), ('runner', 'NN'), ('runs', 'VBZ'), ('the', 'DT'), ('runners', 'NNS'), ('ran', 'VBD'), ('a', 'DT'), ('runner', 'NN'), ('run', 'NN'), ('is', 'VBZ'), ('running', 'VBG'), ('and', 'CC'), ('running', 'VBG'), ('runs', 'NNS')]


## Part C — The Porter stemmer by hand

A faithful re-implementation of Porter (1980) that **logs every rule that fires**. Definitions:
- consonant: a letter other than a, e, i, o, u, and other than *y* preceded by a consonant;
- every word has the form $[C](VC)^m[V]$; $m$ is the **measure**;
- `*v*` stem contains a vowel; `*d` ends in a double consonant; `*o` ends consonant–vowel–consonant where the last consonant is not w, x or y.

In [7]:
VOW = set("aeiou")
def is_cons(w, i):
    if w[i] in VOW: return False
    if w[i] == "y": return i == 0 or not is_cons(w, i - 1)
    return True
def cv_form(w): return "".join("C" if is_cons(w, i) else "V" for i in range(len(w)))
def measure(stem):
    f = cv_form(stem); s = "".join(ch for i, ch in enumerate(f) if i == 0 or f[i-1] != ch)
    return s.count("VC")
def has_vowel(s): return any(not is_cons(s, i) for i in range(len(s)))
def ends_double(s): return len(s) >= 2 and s[-1] == s[-2] and is_cons(s, len(s) - 1)
def cvc(s):
    n = len(s)
    return n >= 3 and is_cons(s, n-3) and not is_cons(s, n-2) and is_cons(s, n-1) and s[-1] not in "wxy"

STEP2 = [("ational","ate"),("tional","tion"),("enci","ence"),("anci","ance"),("izer","ize"),("abli","able"),
         ("alli","al"),("entli","ent"),("eli","e"),("ousli","ous"),("ization","ize"),("ation","ate"),("ator","ate"),
         ("alism","al"),("iveness","ive"),("fulness","ful"),("ousness","ous"),("aliti","al"),("iviti","ive"),("biliti","ble")]
STEP3 = [("icate","ic"),("ative",""),("alize","al"),("iciti","ic"),("ical","ic"),("ful",""),("ness","")]
STEP4 = ["al","ance","ence","er","ic","able","ible","ant","ement","ment","ent","ion","ou","ism","ate","iti","ous","ive","ize"]

def porter(w, log=None):
    L = (lambda *a: log.append(a)) if log is not None else (lambda *a: None)
    if len(w) <= 2: return w
    for a, b in [("sses","ss"),("ies","i"),("ss","ss"),("s","")]:          # step 1a (longest match first)
        if w.endswith(a):
            if a != b: L("1a", f"{a.upper()}->{b.upper() or 'ε'}", w, w[:-len(a)] + b)
            w = w[:-len(a)] + b; break
    extra = False                                                          # step 1b
    if w.endswith("eed"):
        if measure(w[:-3]) > 0: L("1b", "(m>0) EED->EE", w, w[:-1]); w = w[:-1]
    else:
        for a in ("ed", "ing"):
            if w.endswith(a):
                if has_vowel(w[:-len(a)]): L("1b", f"(*v*) {a.upper()}->ε", w, w[:-len(a)]); w = w[:-len(a)]; extra = True
                break
    if extra:
        for a in ("at", "bl", "iz"):
            if w.endswith(a): L("1b'", f"{a.upper()}->{a.upper()}E", w, w + "e"); w += "e"; break
        else:
            if ends_double(w) and w[-1] not in "lsz": L("1b'", "(*d, not L/S/Z) -> single letter", w, w[:-1]); w = w[:-1]
            elif measure(w) == 1 and cvc(w): L("1b'", "(m=1 and *o) -> +E", w, w + "e"); w += "e"
    if w.endswith("y") and has_vowel(w[:-1]): L("1c", "(*v*) Y->I", w, w[:-1] + "i"); w = w[:-1] + "i"
    for name, rules in (("2", STEP2), ("3", STEP3)):                        # steps 2, 3: (m>0)
        hits = [r for r in rules if w.endswith(r[0])]
        if hits:
            a, b = max(hits, key=lambda r: len(r[0])); st = w[:-len(a)]
            if measure(st) > 0: L(name, f"(m={measure(st)}>0) {a.upper()}->{b.upper() or 'ε'}", w, st + b); w = st + b
    hits = [s for s in STEP4 if w.endswith(s)]                              # step 4: (m>1)
    if hits:
        a = max(hits, key=len); st = w[:-len(a)]
        if measure(st) > 1 and (a != "ion" or st[-1:] in ("s", "t")): L("4", f"(m={measure(st)}>1) {a.upper()}->ε", w, st); w = st
    if w.endswith("e"):                                                    # step 5a
        st = w[:-1]; m = measure(st)
        if m > 1 or (m == 1 and not cvc(st)): L("5a", f"(m={m}) E->ε", w, st); w = st
    if measure(w) > 1 and ends_double(w) and w.endswith("l"): L("5b", "(m>1, *d, *L) -> single L", w, w[:-1]); w = w[:-1]
    return w

In [8]:
# Porter's own examples of the measure m
for w in ["tr", "ee", "tree", "y", "by", "trouble", "oats", "trees", "ivy", "troubles", "private", "oaten", "orrery"]:
    print(f"{w:9s} {cv_form(w):9s} m={measure(w)}")

tr        CC        m=0
ee        VV        m=0
tree      CCVV      m=0
y         C         m=0
by        CV        m=0
trouble   CCVVCCV   m=1
oats      VVCC      m=1
trees     CCVVC     m=1
ivy       VCV       m=1
troubles  CCVVCCVC  m=2
private   CCVCVCV   m=2
oaten     VVCVC     m=2
orrery    VCCVCV    m=2


In [9]:
# Step-by-step traces for the eight hand-worked words in the note
for w in ["caresses", "ponies", "hopping", "filing", "relational", "conditional", "generalizations", "oscillators"]:
    log = []; s = porter(w, log)
    print(f"{w} -> {s}")
    for step, rule, before, after in log:
        print(f"    step {step:3s} {rule:32s} {before} -> {after}")

caresses -> caress
    step 1a  SSES->SS                         caresses -> caress
ponies -> poni
    step 1a  IES->I                           ponies -> poni
hopping -> hop
    step 1b  (*v*) ING->ε                     hopping -> hopp
    step 1b' (*d, not L/S/Z) -> single letter hopp -> hop
filing -> file
    step 1b  (*v*) ING->ε                     filing -> fil
    step 1b' (m=1 and *o) -> +E               fil -> file
relational -> relat
    step 2   (m=1>0) ATIONAL->ATE             relational -> relate
    step 5a  (m=2) E->ε                       relate -> relat
conditional -> condit
    step 2   (m=1>0) TIONAL->TION             conditional -> condition
    step 4   (m=2>1) ION->ε                   condition -> condit
generalizations -> gener
    step 1a  S->ε                             generalizations -> generalization
    step 2   (m=3>0) IZATION->IZE             generalization -> generalize
    step 3   (m=2>0) ALIZE->AL                generalize -> general
    step 4   (m=

In [10]:
# Validate the re-implementation against NLTK's ORIGINAL_ALGORITHM mode on every word type in the Gutenberg corpus
P0 = PorterStemmer(mode=PorterStemmer.ORIGINAL_ALGORITHM)
vocab = sorted({w.lower() for f in gutenberg.fileids() for w in gutenberg.words(f) if w.isalpha()})
diff = [(w, porter(w), P0.stem(w)) for w in vocab if porter(w) != P0.stem(w)]
print(f"{len(vocab)} word types, {len(diff)} disagreements:", diff)
print("(all disagreements are words of length <= 2, which Porter's reference C code leaves untouched)")

41487 word types, 8 disagreements: [('as', 'as', 'a'), ('ay', 'ay', 'ai'), ('is', 'is', 'i'), ('ms', 'ms', 'm'), ('os', 'os', 'o'), ('s', 's', ''), ('us', 'us', 'u'), ('vs', 'vs', 'v')]
(all disagreements are words of length <= 2, which Porter's reference C code leaves untouched)


## Part D — Porter vs Snowball vs Lancaster

In [11]:
P = PorterStemmer(); S = SnowballStemmer("english"); La = LancasterStemmer()
words = ["generously", "generalization", "university", "universe", "organization", "organ", "fairly", "sportingly",
         "happiness", "running", "maximum", "presumably", "multiply", "provision", "owed", "ear", "saying", "crying",
         "dying", "news", "sky", "skies", "ran", "geese", "was", "computer", "computational"]
print(f"{'word':16s}{'Porter':14s}{'Snowball':14s}{'Lancaster':14s}")
for w in words:
    print(f"{w:16s}{P.stem(w):14s}{S.stem(w):14s}{La.stem(w):14s}")

word            Porter        Snowball      Lancaster     
generously      gener         generous      gen           
generalization  gener         general       gen           
university      univers       univers       univers       
universe        univers       univers       univers       
organization    organ         organ         org           
organ           organ         organ         org           
fairly          fairli        fair          fair          
sportingly      sportingli    sport         sport         
happiness       happi         happi         happy         
running         run           run           run           
maximum         maximum       maximum       maxim         
presumably      presum        presum        presum        
multiply        multipli      multipli      multiply      
provision       provis        provis        provid        
owed            owe           owe           ow            
ear             ear           ear           ear         

In [12]:
# How aggressive is each stemmer? Distinct stems produced from the 10,000 most frequent Gutenberg word types.
cnt = Counter(w.lower() for f in gutenberg.fileids() for w in gutenberg.words(f) if w.isalpha())
top = [w for w, _ in cnt.most_common(10000)]
for name, st in [("Porter", P), ("Snowball", S), ("Lancaster", La)]:
    stems = [st.stem(w) for w in top]
    print(f"{name:10s} distinct stems = {len(set(stems)):5d}   mean stem length = {np.mean([len(x) for x in stems]):.2f}")
print("mean word length =", round(np.mean([len(w) for w in top]), 2))

Porter     distinct stems =  6680   mean stem length = 5.65
Snowball   distinct stems =  6557   mean stem length = 5.63
Lancaster  distinct stems =  5703   mean stem length = 4.96
mean word length = 6.74


In [13]:
# Over-stemming (different meanings conflated) and under-stemming (same meaning kept apart)
over = [("universe", "university"), ("organ", "organization"), ("policy", "police"), ("general", "generous"), ("new", "news")]
under = [("absorb", "absorption"), ("run", "ran"), ("goose", "geese"), ("alumnus", "alumni"), ("matrix", "matrices")]
for name, pairs in [("over", over), ("under", under)]:
    for a, b in pairs:
        print(f"{name:5s} {a:12s} {b:13s} Porter: {P.stem(a):9s} {P.stem(b):9s} | Lancaster: {La.stem(a):8s} {La.stem(b)}")

over  universe     university    Porter: univers   univers   | Lancaster: univers  univers
over  organ        organization  Porter: organ     organ     | Lancaster: org      org
over  policy       police        Porter: polici    polic     | Lancaster: policy   pol
over  general      generous      Porter: gener     gener     | Lancaster: gen      gen
over  new          news          Porter: new       news      | Lancaster: new      new
under absorb       absorption    Porter: absorb    absorpt   | Lancaster: absorb   absorb
under run          ran           Porter: run       ran       | Lancaster: run      ran
under goose        geese         Porter: goos      gees      | Lancaster: goos     gees
under alumnus      alumni        Porter: alumnu    alumni    | Lancaster: alumn    alumn
under matrix       matrices      Porter: matrix    matric    | Lancaster: matrix   mat


## Part E — POS-dependent lemmatization

`WordNetLemmatizer.lemmatize(word, pos)` assumes **noun** when no POS is given.

In [14]:
L = WordNetLemmatizer()
for w in ["running", "better", "meeting", "leaves", "saw", "was", "ran", "studies", "geese", "feet", "worse", "best", "are", "axes"]:
    print(f"{w:9s} noun:{L.lemmatize(w, 'n'):9s} verb:{L.lemmatize(w, 'v'):9s} adj:{L.lemmatize(w, 'a')}")

running   noun:running   verb:run       adj:running
better    noun:better    verb:better    adj:good
meeting   noun:meeting   verb:meet      adj:meeting
leaves    noun:leaf      verb:leave     adj:leaves
saw       noun:saw       verb:saw       adj:saw
was       noun:wa        verb:be        adj:was
ran       noun:ran       verb:run       adj:ran
studies   noun:study     verb:study     adj:studies
geese     noun:goose     verb:geese     adj:geese
feet      noun:foot      verb:feet      adj:feet
worse     noun:worse     verb:worse     adj:bad
best      noun:best      verb:best      adj:best
are       noun:are       verb:be        adj:are
axes      noun:ax        verb:ax        adj:axes


In [15]:
# Let a POS tagger choose the POS
pos = {"N": wn.NOUN, "V": wn.VERB, "J": wn.ADJ, "R": wn.ADV}
sent = "The leaves fell while she was leaving the meeting; we were meeting better players"
for w, tag in nltk.pos_tag(nltk.word_tokenize(sent)):
    print(f"{w:9s} {tag:4s} -> {L.lemmatize(w.lower(), pos.get(tag[0], wn.NOUN))}")

The       DT   -> the
leaves    NNS  -> leaf
fell      VBD  -> fell
while     IN   -> while
she       PRP  -> she
was       VBD  -> be
leaving   VBG  -> leave
the       DT   -> the
meeting   NN   -> meeting
;         :    -> ;
we        PRP  -> we
were      VBD  -> be
meeting   VBG  -> meet
better    JJR  -> good
players   NNS  -> player


## Part F — Sub-word tokenization: BPE, WordPiece, Unigram

### F1. BPE trainer with a merge log (the Jurafsky & Martin toy corpus)

Counts: low ×5, lowest ×2, newer ×6, wider ×3, new ×2. `_` marks end of word. Ties are broken alphabetically.

In [16]:
def get_stats(vocab):
    pairs = Counter()
    for word, f in vocab.items():
        for a, b in zip(word, word[1:]): pairs[(a, b)] += f
    return pairs

def merge_word(word, pair):
    out, i = [], 0
    while i < len(word):
        if i < len(word) - 1 and (word[i], word[i + 1]) == pair: out.append(word[i] + word[i + 1]); i += 2
        else: out.append(word[i]); i += 1
    return tuple(out)

def train_bpe(counts, k, score="freq", verbose=True):
    vocab = {tuple(w) + ("_",): f for w, f in counts.items()}
    merges = []
    for step in range(1, k + 1):
        st = get_stats(vocab)
        if not st: break
        if score == "freq":
            sc = dict(st)
        else:  # WordPiece-style likelihood score
            uni = Counter()
            for word, f in vocab.items():
                for s in word: uni[s] += f
            sc = {p: c / (uni[p[0]] * uni[p[1]]) for p, c in st.items()}
        top = max(sc.values()); best = sorted(p for p in sc if sc[p] == top)[0]
        vocab = {merge_word(w, best): f for w, f in vocab.items()}
        merges.append(best)
        if verbose:
            print(f"merge {step}: {best[0]!r} + {best[1]!r}  (count {st[best]}, score {sc[best]:.4g})")
            print("         ", {" ".join(w): f for w, f in vocab.items()})
    return merges, vocab

def encode(word, merges):
    w = tuple(word) + ("_",)
    for p in merges: w = merge_word(w, p)    # apply merges in the order they were learned
    return list(w)

corpus = {"low": 5, "lowest": 2, "newer": 6, "wider": 3, "new": 2}
init = {tuple(w) + ("_",): f for w, f in corpus.items()}
print("initial symbols:", sorted({s for w in init for s in w}))
print("initial pair counts:", sorted(get_stats(init).items(), key=lambda kv: -kv[1]))
merges, _ = train_bpe(corpus, 8)

initial symbols: ['_', 'd', 'e', 'i', 'l', 'n', 'o', 'r', 's', 't', 'w']
initial pair counts: [(('e', 'r'), 9), (('r', '_'), 9), (('w', 'e'), 8), (('n', 'e'), 8), (('e', 'w'), 8), (('l', 'o'), 7), (('o', 'w'), 7), (('w', '_'), 7), (('w', 'i'), 3), (('i', 'd'), 3), (('d', 'e'), 3), (('e', 's'), 2), (('s', 't'), 2), (('t', '_'), 2)]
merge 1: 'e' + 'r'  (count 9, score 9)
          {'l o w _': 5, 'l o w e s t _': 2, 'n e w er _': 6, 'w i d er _': 3, 'n e w _': 2}
merge 2: 'er' + '_'  (count 9, score 9)
          {'l o w _': 5, 'l o w e s t _': 2, 'n e w er_': 6, 'w i d er_': 3, 'n e w _': 2}
merge 3: 'e' + 'w'  (count 8, score 8)
          {'l o w _': 5, 'l o w e s t _': 2, 'n ew er_': 6, 'w i d er_': 3, 'n ew _': 2}
merge 4: 'n' + 'ew'  (count 8, score 8)
          {'l o w _': 5, 'l o w e s t _': 2, 'new er_': 6, 'w i d er_': 3, 'new _': 2}
merge 5: 'l' + 'o'  (count 7, score 7)
          {'lo w _': 5, 'lo w e s t _': 2, 'new er_': 6, 'w i d er_': 3, 'new _': 2}
merge 6: 'lo' + 'w'  (cou

In [17]:
for w in ["lower", "newest", "widest", "low", "renew"]:
    print(f"{w:7s} after 5 merges: {encode(w, merges[:5])}   after 8 merges: {encode(w, merges)}")

lower   after 5 merges: ['lo', 'w', 'er_']   after 8 merges: ['low', 'er_']
newest  after 5 merges: ['new', 'e', 's', 't', '_']   after 8 merges: ['new', 'e', 's', 't', '_']
widest  after 5 merges: ['w', 'i', 'd', 'e', 's', 't', '_']   after 8 merges: ['w', 'i', 'd', 'e', 's', 't', '_']
low     after 5 merges: ['lo', 'w', '_']   after 8 merges: ['low_']
renew   after 5 merges: ['r', 'e', 'new', '_']   after 8 merges: ['r', 'e', 'new', '_']


### F2. WordPiece's merge criterion on the same corpus

WordPiece picks the pair maximising $\dfrac{\text{count}(ab)}{\text{count}(a)\,\text{count}(b)}$ (how much more often $a,b$ co-occur than chance), not the raw count.

In [18]:
wp_merges, _ = train_bpe(corpus, 3, score="wp")

merge 1: 's' + 't'  (count 2, score 0.5)
          {'l o w _': 5, 'l o w e st _': 2, 'n e w e r _': 6, 'w i d e r _': 3, 'n e w _': 2}
merge 2: 'i' + 'd'  (count 3, score 0.3333)
          {'l o w _': 5, 'l o w e st _': 2, 'n e w e r _': 6, 'w id e r _': 3, 'n e w _': 2}
merge 3: 'l' + 'o'  (count 7, score 0.1429)
          {'lo w _': 5, 'lo w e st _': 2, 'n e w e r _': 6, 'w id e r _': 3, 'n e w _': 2}


### F3. Unigram LM tokenization: Viterbi over a toy vocabulary

A Unigram tokenizer keeps a vocabulary with probabilities and picks the segmentation with the highest product of piece probabilities (dynamic programming over end positions).

In [19]:
import math
probs = {"u": .02, "n": .03, "a": .04, "v": .01, "i": .03, "l": .03, "b": .01, "e": .05,
         "un": .05, "avail": .02, "able": .04, "availab": .001, "le": .02, "unavail": .0005}
def viterbi(word, probs):
    best = [(0.0, None)] + [(-math.inf, None)] * len(word)   # best[i] = (log-prob, start of last piece)
    for end in range(1, len(word) + 1):
        for start in range(end):
            piece = word[start:end]
            if piece in probs and best[start][0] + math.log(probs[piece]) > best[end][0]:
                best[end] = (best[start][0] + math.log(probs[piece]), start)
    pieces, i = [], len(word)
    while i > 0:
        s = best[i][1]; pieces.append(word[s:i]); i = s
    return pieces[::-1], math.exp(best[-1][0])
seg, p = viterbi("unavailable", probs)
print(seg, f"P = {p:.3g}")
for alt in [["unavail", "able"], ["un", "availab", "le"], ["un", "avail", "able"]]:
    print(alt, f"P = {np.prod([probs[x] for x in alt]):.3g}")

['un', 'avail', 'able'] P = 4e-05
['unavail', 'able'] P = 2e-05
['un', 'availab', 'le'] P = 1e-06
['un', 'avail', 'able'] P = 4e-05


### F4. Vocabulary size vs sequence length, and the language-disparity effect

Train a **byte-level** BPE (symbols are UTF-8 bytes, as in GPT-2-style tokenizers) on English text only, then count tokens for an English sentence and its Hindi translation.

In [20]:
def train_byte_bpe(text, k):
    counts = Counter(re.findall(r"\S+", text))
    vocab = {tuple(bytes([b]) for b in w.encode("utf-8")): f for w, f in counts.items()}
    merges = []
    for _ in range(k):
        st = get_stats(vocab)
        if not st: break
        best = max(st, key=st.get); merges.append(best)
        vocab = {merge_word(w, best): f for w, f in vocab.items()}
    return merges

def byte_encode(text, merges):
    out = []
    for w in re.findall(r"\S+", text):
        sym = tuple(bytes([b]) for b in w.encode("utf-8"))
        for p in merges: sym = merge_word(sym, p)
        out.extend(sym)
    return out

eng_train = gutenberg.raw("austen-emma.txt")[:300000]
bm = train_byte_bpe(eng_train, 500)
en = "India is a country with many languages and scripts."
hi = "भारत अनेक भाषाओं और लिपियों वाला देश है।"
for name, s in [("English", en), ("Hindi", hi)]:
    toks = byte_encode(s, bm)
    print(f"{name:8s} chars={len(s):3d} utf8 bytes={len(s.encode()):3d} BPE tokens={len(toks):3d} tokens/char={len(toks)/len(s):.2f}")

English  chars= 51 utf8 bytes= 51 BPE tokens= 25 tokens/char=0.49
Hindi    chars= 40 utf8 bytes=106 BPE tokens= 99 tokens/char=2.48


## Part G — Levenshtein distance with backtrace

In [21]:
def levenshtein(a, b, sub_cost=1, show=False):
    n, m = len(a), len(b)
    D = np.zeros((n + 1, m + 1), dtype=int)
    D[:, 0] = np.arange(n + 1); D[0, :] = np.arange(m + 1)
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            D[i, j] = min(D[i-1, j] + 1,                                        # deletion
                          D[i, j-1] + 1,                                        # insertion
                          D[i-1, j-1] + (0 if a[i-1] == b[j-1] else sub_cost))  # substitution / copy
    i, j, ops = n, m, []
    while i > 0 or j > 0:                                                       # backtrace
        if i > 0 and j > 0 and D[i, j] == D[i-1, j-1] + (0 if a[i-1] == b[j-1] else sub_cost):
            ops.append(("copy" if a[i-1] == b[j-1] else f"sub {a[i-1]}->{b[j-1]}")); i, j = i - 1, j - 1
        elif i > 0 and D[i, j] == D[i-1, j] + 1:
            ops.append(f"del {a[i-1]}"); i -= 1
        else:
            ops.append(f"ins {b[j-1]}"); j -= 1
    if show:
        print("      " + "  ".join("#" + b))
        for r, ch in enumerate("#" + a): print(f"   {ch}  " + "  ".join(str(x) for x in D[r]))
    return int(D[n, m]), ops[::-1]

d, ops = levenshtein("kitten", "sitting", show=True)
print("distance", d, [o for o in ops if o != "copy"])
for a, b, sc in [("intention", "execution", 1), ("intention", "execution", 2), ("sunday", "saturday", 1),
                 ("flaw", "lawn", 1), ("teh", "the", 1), ("acress", "actress", 1), ("rain", "shine", 1)]:
    d, ops = levenshtein(a, b, sc)
    print(f"{a:10s} -> {b:10s} sub_cost={sc}: distance {d}  edits {[o for o in ops if o != 'copy']}")

      #  s  i  t  t  i  n  g
   #  0  1  2  3  4  5  6  7
   k  1  1  2  3  4  5  6  7
   i  2  2  1  2  3  4  5  6
   t  3  3  2  1  2  3  4  5
   t  4  4  3  2  1  2  3  4
   e  5  5  4  3  2  2  3  4
   n  6  6  5  4  3  3  2  3
distance 3 ['sub k->s', 'sub e->i', 'ins g']
intention  -> execution  sub_cost=1: distance 5  edits ['sub i->e', 'sub n->x', 'sub t->e', 'sub e->c', 'sub n->u']
intention  -> execution  sub_cost=2: distance 8  edits ['del i', 'sub n->e', 'sub t->x', 'ins c', 'sub n->u']
sunday     -> saturday   sub_cost=1: distance 3  edits ['ins a', 'ins t', 'sub n->r']
flaw       -> lawn       sub_cost=1: distance 2  edits ['del f', 'ins n']
teh        -> the        sub_cost=1: distance 2  edits ['sub e->h', 'sub h->e']
acress     -> actress    sub_cost=1: distance 1  edits ['ins t']
rain       -> shine      sub_cost=1: distance 3  edits ['sub r->s', 'sub a->h', 'ins e']


In [22]:
def damerau_osa(a, b):
    "optimal string alignment: Levenshtein + adjacent transposition (cost 1)"
    n, m = len(a), len(b)
    D = np.zeros((n + 1, m + 1), dtype=int); D[:, 0] = range(n + 1); D[0, :] = range(m + 1)
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            D[i, j] = min(D[i-1, j] + 1, D[i, j-1] + 1, D[i-1, j-1] + (a[i-1] != b[j-1]))
            if i > 1 and j > 1 and a[i-1] == b[j-2] and a[i-2] == b[j-1]:
                D[i, j] = min(D[i, j], D[i-2, j-2] + 1)
    return int(D[n, m])
for a, b in [("teh", "the"), ("recieve", "receive"), ("form", "from")]:
    print(a, b, "Levenshtein", levenshtein(a, b)[0], "Damerau(OSA)", damerau_osa(a, b))

teh the Levenshtein 2 Damerau(OSA) 1
recieve receive Levenshtein 2 Damerau(OSA) 1
form from Levenshtein 2 Damerau(OSA) 1


In [23]:
# A tiny Norvig-style spelling corrector: candidates within edit distance 1 (then 2), ranked by corpus frequency.
WORDS = Counter(w for f in gutenberg.fileids() for w in tokens(f))
LETTERS = "abcdefghijklmnopqrstuvwxyz"
def edits1(w):
    splits = [(w[:i], w[i:]) for i in range(len(w) + 1)]
    return set([L + R[1:] for L, R in splits if R] + [L + R[1] + R[0] + R[2:] for L, R in splits if len(R) > 1] +
               [L + c + R[1:] for L, R in splits if R for c in LETTERS] + [L + c + R for L, R in splits for c in LETTERS])
def known(ws): return {w for w in ws if w in WORDS}
def correct(w):
    cands = known([w]) or known(edits1(w)) or known(e2 for e1 in edits1(w) for e2 in edits1(e1)) or {w}
    return max(cands, key=WORDS.get)
print("size of edits1('speling'):", len(edits1("speling")))
for w in ["speling", "korrect", "teh", "whaal", "captian", "recieve", "abuot"]:
    print(f"{w:8s} -> {correct(w)}")

size of edits1('speling'): 390
speling  -> spelling
korrect  -> correct
teh      -> the
whaal    -> shall
captian  -> captain
recieve  -> receive
abuot    -> about


## Part H — Unicode normalisation (incl. Devanagari)

In [24]:
def cps(s): return " ".join(f"U+{ord(ch):04X}" for ch in s)
samples = [("é precomposed", "é"), ("e + combining acute", "é"), ("Angstrom sign", "Å"),
           ("ligature fi", "ﬁ"), ("circled one", "①"), ("superscript two", "x²"),
           ("full-width Hello", "Ｈｅｌｌｏ"), ("क़ precomposed U+0958", "क़"), ("क + nukta", "क़")]
for name, s in samples:
    print(f"{name:22s} {cps(s):28s}| NFC {cps(unicodedata.normalize('NFC', s)):16s}| NFD {cps(unicodedata.normalize('NFD', s)):16s}| NFKC {unicodedata.normalize('NFKC', s)!r}")
print("'café' (precomposed) == 'café' (decomposed)?", "café" == "café",
      "| after NFC:", unicodedata.normalize("NFC", "café") == unicodedata.normalize("NFC", "café"))

é precomposed          U+00E9                      | NFC U+00E9          | NFD U+0065 U+0301   | NFKC 'é'
e + combining acute    U+0065 U+0301               | NFC U+00E9          | NFD U+0065 U+0301   | NFKC 'é'
Angstrom sign          U+212B                      | NFC U+00C5          | NFD U+0041 U+030A   | NFKC 'Å'
ligature fi            U+FB01                      | NFC U+FB01          | NFD U+FB01          | NFKC 'fi'
circled one            U+2460                      | NFC U+2460          | NFD U+2460          | NFKC '1'
superscript two        U+0078 U+00B2               | NFC U+0078 U+00B2   | NFD U+0078 U+00B2   | NFKC 'x2'
full-width Hello       U+FF28 U+FF45 U+FF4C U+FF4C U+FF4F| NFC U+FF28 U+FF45 U+FF4C U+FF4C U+FF4F| NFD U+FF28 U+FF45 U+FF4C U+FF4C U+FF4F| NFKC 'Hello'
क़ precomposed U+0958  U+0958                      | NFC U+0915 U+093C   | NFD U+0915 U+093C   | NFKC 'क़'
क + nukta              U+0915 U+093C               | NFC U+0915 U+093C   | NFD U+0915 U+093C   | NFKC '

In [25]:
for s in ["नमस्ते", "हिन्दी", "ಕನ್ನಡ", "क्ष", "क्‍ष", "क्‌ष"]:
    print(f"{s!r:14s} code points={len(s)} utf8 bytes={len(s.encode())}  " + ", ".join(unicodedata.name(ch) for ch in s))
# ZWJ / ZWNJ survive every normalisation form; they must be handled explicitly
z = "क्‍ष"
print({f: len(unicodedata.normalize(f, z)) for f in ["NFC", "NFD", "NFKC", "NFKD"]})
print("strip ZW(N)J:", cps(re.sub("[‌‍]", "", z)))

'नमस्ते'       code points=6 utf8 bytes=18  DEVANAGARI LETTER NA, DEVANAGARI LETTER MA, DEVANAGARI LETTER SA, DEVANAGARI SIGN VIRAMA, DEVANAGARI LETTER TA, DEVANAGARI VOWEL SIGN E
'हिन्दी'       code points=6 utf8 bytes=18  DEVANAGARI LETTER HA, DEVANAGARI VOWEL SIGN I, DEVANAGARI LETTER NA, DEVANAGARI SIGN VIRAMA, DEVANAGARI LETTER DA, DEVANAGARI VOWEL SIGN II
'ಕನ್ನಡ'        code points=5 utf8 bytes=15  KANNADA LETTER KA, KANNADA LETTER NA, KANNADA SIGN VIRAMA, KANNADA LETTER NA, KANNADA LETTER DDA
'क्ष'          code points=3 utf8 bytes=9  DEVANAGARI LETTER KA, DEVANAGARI SIGN VIRAMA, DEVANAGARI LETTER SSA
'क्\u200dष'    code points=4 utf8 bytes=12  DEVANAGARI LETTER KA, DEVANAGARI SIGN VIRAMA, ZERO WIDTH JOINER, DEVANAGARI LETTER SSA
'क्\u200cष'    code points=4 utf8 bytes=12  DEVANAGARI LETTER KA, DEVANAGARI SIGN VIRAMA, ZERO WIDTH NON-JOINER, DEVANAGARI LETTER SSA
{'NFC': 4, 'NFD': 4, 'NFKC': 4, 'NFKD': 4}
strip ZW(N)J: U+0915 U+094D U+0937


In [26]:
# Script detection by Unicode block: a first step for code-mixed text
def script(tok):
    names = {unicodedata.name(ch, "?").split()[0] for ch in tok if ch.isalpha()}
    return "/".join(sorted(names)) or "OTHER"
for tok in "yaar ये movie बहुत achhi थी ಕನ್ನಡ 2024".split():
    print(f"{tok:8s} {script(tok)}")

yaar     LATIN
ये       DEVANAGARI
movie    LATIN
बहुत     DEVANAGARI
achhi    LATIN
थी       DEVANAGARI
ಕನ್ನಡ    KANNADA
2024     OTHER


## Part I — A regex tokenizer with unit tests

Order matters in the alternation: specific patterns (URL, e-mail, price, date, abbreviation) must come **before** the generic word pattern.

In [27]:
# Pitfall first: Python's \w does NOT match Indic vowel signs and virama (Unicode categories Mn/Mc)
print(re.findall(r"\w+", "नमस्ते दुनिया"))
print([(ch, unicodedata.category(ch)) for ch in "नमस्ते"])

['नमस', 'त', 'द', 'न', 'य']
[('न', 'Lo'), ('म', 'Lo'), ('स', 'Lo'), ('्', 'Mn'), ('त', 'Lo'), ('े', 'Mn')]


In [28]:
W = r"[\w\u0900-\u0963\u0966-\u0DFF\u200c\u200d]"    # \w + Indic blocks (Devanagari..Sinhala) minus the dandas U+0964/65, + ZWNJ/ZWJ
TOKEN_RE = re.compile(r'''
    https?://\S+                       # URLs
  | [\w.+-]+@[\w-]+(?:\.[\w-]+)+       # e-mail addresses
  | [#@]\w+                            # hashtags, mentions
  | [$₹]\d+(?:[.,]\d+)*                # prices
  | \d{1,2}/\d{1,2}/\d{2,4}            # dates
  | \d+(?:[.,]\d+)*%?                  # numbers, decimals, percentages
  | (?:[A-Za-z]\.){2,}                 # abbreviations such as U.S.A. and Ph.D. style initials
  | W+(?:[-'&]W+)*                     # words incl. hyphens, apostrophes (we're), AT&T, Indic marks
  | \.\.\.|[^\w\s]                     # ellipsis or any single punctuation mark
'''.replace("W", W), re.VERBOSE)
tok = TOKEN_RE.findall
tests = {
    "We're paying $45.55 on 01/02/06.": ["We're", "paying", "$45.55", "on", "01/02/06", "."],
    "Mail someone@cs.colorado.edu or see http://www.stanford.edu": ["Mail", "someone@cs.colorado.edu", "or", "see", "http://www.stanford.edu"],
    "#nlproc rocks at 4.3% growth": ["#nlproc", "rocks", "at", "4.3%", "growth"],
    "The U.S.A. and AT&T, state-of-the-art...": ["The", "U.S.A.", "and", "AT&T", ",", "state-of-the-art", "..."],
    "Fee: ₹1,20,000 only!": ["Fee", ":", "₹1,20,000", "only", "!"],
    "नमस्ते दुनिया, hello!": ["नमस्ते", "दुनिया", ",", "hello", "!"],
    "भारत महान है।": ["भारत", "महान", "है", "।"],
}
for s, expected in tests.items():
    got = tok(s)
    print("PASS" if got == expected else "FAIL", got)
    assert got == expected

PASS ["We're", 'paying', '$45.55', 'on', '01/02/06', '.']
PASS ['Mail', 'someone@cs.colorado.edu', 'or', 'see', 'http://www.stanford.edu']
PASS ['#nlproc', 'rocks', 'at', '4.3%', 'growth']
PASS ['The', 'U.S.A.', 'and', 'AT&T', ',', 'state-of-the-art', '...']
PASS ['Fee', ':', '₹1,20,000', 'only', '!']
PASS ['नमस्ते', 'दुनिया', ',', 'hello', '!']
PASS ['भारत', 'महान', 'है', '।']


In [29]:
# The same sentences through NLTK's word_tokenize for comparison
from nltk.tokenize import word_tokenize
for s in list(tests)[:4]:
    print(word_tokenize(s))

['We', "'re", 'paying', '$', '45.55', 'on', '01/02/06', '.']
['Mail', 'someone', '@', 'cs.colorado.edu', 'or', 'see', 'http', ':', '//www.stanford.edu']
['#', 'nlproc', 'rocks', 'at', '4.3', '%', 'growth']
['The', 'U.S.A.', 'and', 'AT', '&', 'T', ',', 'state-of-the-art', '...']


## Part J — Tricky sentence segmentation

In [30]:
from nltk.tokenize import sent_tokenize
from nltk.tokenize.punkt import PunktSentenceTokenizer, PunktParameters
t = ('Dr. A. P. J. Abdul Kalam visited IIIT Dharwad at 10 a.m. on Monday. He said, "Dream big!" '
     'The fee is Rs. 4.5 lakh, i.e. about 5.4k USD. Visit www.iiitdwd.ac.in. Was it worth it? Yes.')
naive = re.split(r"(?<=[.!?])\s+", t)
print(len(naive), "naive pieces:")
for s in naive: print("   |", s)
punkt = sent_tokenize(t)
print(len(punkt), "Punkt sentences:")
for s in punkt: print("   |", s)
pp = PunktParameters(); pp.abbrev_types = {"dr", "rs", "i.e", "a.m", "a", "p", "j"}
custom = PunktSentenceTokenizer(pp).tokenize(t)
print(len(custom), "Punkt + domain abbreviation list:")
for s in custom: print("   |", s)

12 naive pieces:
   | Dr.
   | A.
   | P.
   | J.
   | Abdul Kalam visited IIIT Dharwad at 10 a.m.
   | on Monday.
   | He said, "Dream big!" The fee is Rs.
   | 4.5 lakh, i.e.
   | about 5.4k USD.
   | Visit www.iiitdwd.ac.in.
   | Was it worth it?
   | Yes.
8 Punkt sentences:
   | Dr. A. P. J. Abdul Kalam visited IIIT Dharwad at 10 a.m. on Monday.
   | He said, "Dream big!"
   | The fee is Rs.
   | 4.5 lakh, i.e.
   | about 5.4k USD.
   | Visit www.iiitdwd.ac.in.
   | Was it worth it?
   | Yes.
6 Punkt + domain abbreviation list:
   | Dr. A. P. J. Abdul Kalam visited IIIT Dharwad at 10 a.m. on Monday.
   | He said, "Dream big!"
   | The fee is Rs. 4.5 lakh, i.e. about 5.4k USD.
   | Visit www.iiitdwd.ac.in.
   | Was it worth it?
   | Yes.
